In [0]:
%uv pip install databricks-feature-engineering
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from databricks.feature_engineering import FeatureEngineeringClient
import logging


In [0]:
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s - %(name)s - %(levelname)s - %(message)s"
)
logger = logging.getLogger("DemandFeatureStore")

In [0]:
class Features:
    def __init__( self,spark,feature_client,gold_table,feature_table,label_table):

        self.spark = spark
        self.feature_client = feature_client
        self.gold_table = gold_table
        self.feature_table_name = feature_table
        self.label_table_name = label_table

        logger.info("Features class has instantiated")
        logger.info("Feature table Name: %s",self.feature_table_name)
        logger.info("Label table: %s",self.label_table_name)

    def read_gold_table(self):
        logger.info("Reading Gold feature table")
        df = self.spark.table(self.gold_table)
        logger.info("Gold feature table loaded successfully with %d columns",len(df.columns))
        return df

    def create_primary_key(self,df):
        logger.info("Creating primary key for feature table")
        window = Window.partitionBy("transaction_date")\
            .orderBy("product_name","destination_city")
        
        df = df.withColumn("row_number",F.row_number().over(window))
        df = df.withColumn("primary_key",\
            F.concat_ws("-",\
                F.date_format(F.col("transaction_date"),"yyyyMMdd"),F.col("row_number")))
        df = df.drop("row_number")
        logger.info("Primary key has been created")
        return df
    
    def split_features(self, df):

        logger.info("Splitting features and target")
        feature_columns = [
            "primary_key","transaction_date","product_name","destination_city","avg_unit_price","total_inventory",
            "transaction_count","demand_1","demand_7","rolling_7_day_avg","month","day_of_week"]

        label_columns = ["primary_key","total_demand"]
        X = df.select(feature_columns)
        y = df.select(label_columns)
        logger.info("Features df created")
        logger.info("Label df is created")
        return X, y
    
    def load_features(self, X):
        logger.info("Loading features into Feature Store: %s",self.feature_table_name)
        self.feature_client.create_table(
            name=self.feature_table_name,
            primary_keys=["primary_key"],
            df=X,
            description="Feature table for demand prediction"
        )
        logger.info("Feature Store table created successfully")
    
    def load_labels(self, y):
        logger.info("Loading labels into table: %s",self.label_table_name)
        (
            y.write
            .format("delta")
            .mode("overwrite")
            .saveAsTable(self.label_table_name)
        )
        logger.info("Label table created successfully")
    
    def run(self):

        logger.info("Feature Store Process")
        gold_df = self.read_gold_table()
        gold_df = self.create_primary_key(gold_df)
        X, y = self.split_features(gold_df)
        self.load_features(X)
        self.load_labels(y)
        logger.info("Process Complete Feature Store Created")
        return True
        



In [0]:
fe = FeatureEngineeringClient()
catalog = "oag"
features = Features(
    spark=spark,
    feature_client=fe,
    gold_table=f"{catalog}.gold.gold_features",
    feature_table=f"{catalog}.gold.demand_input_features",
    label_table=f"{catalog}.gold.demand_output"
)
features.run()